# Lesson 7: Green Cover and Built-up Areas

**Planning question: Where is green space lacking?**

Green cover cools cities, absorbs rain and improves health. Bengaluru was once called the Garden City,
but much of its tree cover has been lost. In this notebook you will:

1. Calculate **NDVI**, a measure of green vegetation, from a Sentinel-2 image
2. Calculate **NDBI**, a measure of built-up surfaces
3. Summarise both for each BBMP ward (**zonal statistics**)
4. Find the wards with the least green cover

Part of [RS & GIS for Spatial Planners](https://soniadas123.github.io/rs-gis-spatial-planner/).

In [ ]:
!pip install -q pystac-client planetary-computer odc-stac rioxarray

In [ ]:
import pystac_client
import planetary_computer
import odc.stac
import rioxarray
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rasterio.features import rasterize
from shapely.geometry import shape, box
from shapely import union_all

In [ ]:
WARDS_URL = "https://raw.githubusercontent.com/datameet/Municipal_Spatial_Data/master/Bangalore/BBMP.geojson"
DATES = "2024-01-01/2024-03-31"
UTM_CRS = "EPSG:32643"
RESOLUTION = 20   # metres

## Step 1: Load the wards

The box we load from the satellite is the area covered by all BBMP wards.

In [ ]:
wards = gpd.read_file(WARDS_URL)
BBOX = list(wards.total_bounds)
print("Wards:", len(wards), " Box:", [round(v, 3) for v in BBOX])

## Step 2: Find and load a clear Sentinel-2 image

This works the same way as in Lesson 6. We also load **SCL**, the scene classification layer,
which marks clouds and cloud shadows so we can remove them.

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
items = catalog.search(collections=["sentinel-2-l2a"], bbox=BBOX, datetime=DATES,
                       query={"eo:cloud_cover": {"lt": 10}}).item_collection()


def clearest_date_items(items, bbox):
    """Return the items from the least cloudy date that fully covers bbox."""
    by_date = {}
    for item in items:
        by_date.setdefault(item.datetime.date(), []).append(item)
    options = []
    for date, date_items in by_date.items():
        if union_all([shape(i.geometry) for i in date_items]).contains(box(*bbox)):
            cloud = np.mean([i.properties["eo:cloud_cover"] for i in date_items])
            options.append((cloud, date, date_items))
    if not options:
        raise ValueError("No date covers the whole box. Try a longer date range or a higher cloud limit.")
    cloud, date, date_items = min(options, key=lambda option: option[0])
    print(f"Using {date} ({len(date_items)} tiles, {cloud:.1f}% cloud)")
    return date_items


best_items = clearest_date_items(items, BBOX)

ds = odc.stac.load(best_items, bands=["B04", "B08", "B11", "SCL"], bbox=BBOX,
                   crs=UTM_CRS, resolution=RESOLUTION, groupby="solar_day").isel(time=0)

## Step 3: Reflectance and cloud mask

We convert the bands to reflectance (see Lesson 6), then keep only pixels whose SCL class is
vegetation (4), not vegetated (5) or water (6). Clouds, shadows and missing data become empty (NaN).

In [ ]:
offset = 1000 if float(best_items[0].properties["s2:processing_baseline"]) >= 4 else 0
clear = ds["SCL"].isin([4, 5, 6])

red = ((ds["B04"].astype("float32") - offset) / 10000).where(clear)
nir = ((ds["B08"].astype("float32") - offset) / 10000).where(clear)
swir = ((ds["B11"].astype("float32") - offset) / 10000).where(clear)
print(f"Clear pixels: {float(clear.mean()) * 100:.1f}%")

## Step 4: NDVI, the vegetation index

$$NDVI = \frac{NIR - Red}{NIR + Red}$$

Healthy plants reflect a lot of near-infrared and absorb red light, so they have a **high** NDVI.

| NDVI | Usually means |
|---|---|
| below 0 | Water |
| 0 to 0.2 | Buildings, roads, bare soil |
| 0.2 to 0.4 | Grass, scattered trees |
| above 0.4 | Dense trees and vegetation |

In [ ]:
ndvi = (nir - red) / (nir + red)

ndvi.plot(cmap="RdYlGn", vmin=-0.2, vmax=0.8, size=9,
          cbar_kwargs={"label": "NDVI"})
wards.to_crs(UTM_CRS).boundary.plot(ax=plt.gca(), color="black", linewidth=0.2)
plt.title("NDVI: green cover across Bengaluru")
plt.show()

## Step 5: NDBI, the built-up index

$$NDBI = \frac{SWIR - NIR}{SWIR + NIR}$$

Roofs, roads and bare ground reflect more short-wave infrared than near-infrared, so built-up areas have a **high** NDBI.

In [ ]:
ndbi = (swir - nir) / (swir + nir)

ndbi.plot(cmap="magma", vmin=-0.3, vmax=0.3, size=9, cbar_kwargs={"label": "NDBI"})
plt.title("NDBI: built-up surfaces")
plt.show()

## Step 6: How green is the city?

We sort every pixel into the NDVI classes from the table above and count them.

In [ ]:
classes = pd.cut(ndvi.values.ravel(), bins=[-1, 0, 0.2, 0.4, 1],
                 labels=["Water", "Built-up / bare", "Sparse vegetation", "Dense vegetation"])
shares = pd.Series(classes).value_counts(normalize=True).mul(100).round(1)
shares

## Step 7: Zonal statistics by ward

To average NDVI inside each ward, we first **rasterize** the wards: we burn each ward's number into a grid that matches the satellite image.
Then we group the pixels by ward number and take the mean, just as you would with a pivot table.

In [ ]:
wards_utm = wards.to_crs(UTM_CRS).reset_index(drop=True)
wards_utm["zone"] = wards_utm.index + 1

zones = rasterize(
    zip(wards_utm.geometry, wards_utm["zone"]),
    out_shape=ndvi.shape,
    transform=ndvi.rio.transform(),
    fill=0,
    dtype="int32",
)

pixels = pd.DataFrame({
    "zone": zones.ravel(),
    "ndvi": ndvi.values.ravel(),
    "ndbi": ndbi.values.ravel(),
}).dropna()
pixels = pixels[pixels["zone"] > 0]
pixels["green"] = pixels["ndvi"] > 0.4

ward_stats = pixels.groupby("zone").agg(
    mean_ndvi=("ndvi", "mean"),
    mean_ndbi=("ndbi", "mean"),
    green_cover_pct=("green", "mean"),
)
ward_stats["green_cover_pct"] *= 100

wards_utm = wards_utm.merge(ward_stats, left_on="zone", right_index=True, how="left")
wards_utm[["KGISWardName", "mean_ndvi", "mean_ndbi", "green_cover_pct"]].round(2).head()

## Step 8: Map green cover by ward

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
wards_utm.plot(ax=axes[0], column="green_cover_pct", cmap="Greens", legend=True,
               edgecolor="grey", linewidth=0.2, legend_kwds={"label": "Dense green cover (%)", "shrink": 0.6})
axes[0].set_title("Share of dense vegetation per ward")
wards_utm.plot(ax=axes[1], column="mean_ndbi", cmap="magma", legend=True,
               edgecolor="grey", linewidth=0.2, legend_kwds={"label": "Mean NDBI", "shrink": 0.6})
axes[1].set_title("Built-up intensity per ward")
for ax in axes:
    ax.set_axis_off()
plt.show()

## Step 9: The least green wards

These are the wards where new parks and street trees would make the biggest difference.

In [ ]:
wards_utm.nsmallest(15, "green_cover_pct")[["KGISWardName", "green_cover_pct", "mean_ndvi"]].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(wards_utm["mean_ndbi"], wards_utm["green_cover_pct"], s=12, color="teal")
ax.set_xlabel("Mean NDBI (more built-up to the right)")
ax.set_ylabel("Dense green cover (%)")
ax.set_title("More built-up wards have less green cover")
plt.show()

## Step 10: Save the results

In [ ]:
result = wards_utm[["KGISWardName", "mean_ndvi", "mean_ndbi", "green_cover_pct", "geometry"]].round(3)
result.to_file("bengaluru_green_cover.gpkg", driver="GPKG")
print("Saved bengaluru_green_cover.gpkg and bengaluru_ndvi.tif")
ndvi.rio.to_raster("bengaluru_ndvi.tif")

## What this means for planning

- Ward-level green cover gives clear, comparable evidence for where to put greening budgets.
- NDVI picks up all vegetation, including private gardens and empty plots with weeds.
  It shows green cover, not public open space. Combine it with the park access from Lesson 4.
- A single image shows one moment in time. Seasons change NDVI, so compare images from the same months each year.

## Try it yourself

1. Change `DATES` to `"2019-01-01/2019-03-31"` and compare green cover then and now.
2. Change the dense vegetation threshold from 0.4 to 0.3. How sensitive is the ranking?
3. Find your own ward with `wards_utm[wards_utm["KGISWardName"].str.contains("Koramangala")]`.

---
*Enjoying the course? Support it with a [star on GitHub](https://github.com/soniadas123/rs-gis-spatial-planner).*